# Notebook 28 - Diagnosing the recovery collapse

NB27 logged 25 collapsed evaluations in 350, and 4 of the 7 collapse events struck **all five selection methods at the identical epoch**. This notebook identifies the mechanism.

**Two hypotheses**

- **H1, BatchNorm running statistics.** The weights are still usable; the stored running mean and variance have been displaced, so evaluation collapses while the same weights evaluated with batch statistics stay healthy.
- **H2, weight damage.** The optimisation step moved the weights to a bad point, so the model is collapsed under both evaluation modes.

**Discriminator.** At every epoch the same weights are evaluated twice, once in eval mode (running statistics) and once with batch statistics. H1 predicts a large gap between the two; H2 predicts both collapse together. Per-epoch BatchNorm statistic norms, per-batch losses and batch class composition are logged alongside. Unit 0, the raw pruned model, serves as a free positive control.

The batch-statistic probe saves and restores every BatchNorm buffer, so it cannot alter a model; stage 4 proves this at runtime, and also proves the probe differs from eval mode, before any result is produced.

Reproduces NB27 cells that collapsed (shallow seed 307, deep seed 401) plus a clean control (shallow seed 101), with identical frozen structures, recipe and seeds. Frozen 40% structures. No new selection. No test access.

Stages: 1 bootstrap, 2 hypotheses and targets, 3 data and teachers, 4 dual-mode helpers, 5 reproduction runs (resumable; interrupted cells are discarded and redone), 6 verdict, 7 figures.

In [ ]:
# Stage 1 - bootstrap and imports
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, copy
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, yaml
import matplotlib.pyplot as plt

REPO = Path("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
assert REPO.exists(), f"repo not found: {REPO}"
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.saber.bridge_ciciot import load_bridge
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.surgery import prune_cnn1d_channels
from src.saber.metrics import full_model_audit, action_weighted_boundary_inversion_rate

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"
OUT = R / "28_collapse_diagnosis"
OUT.mkdir(parents=True, exist_ok=True)
print("repo:", REPO)
print("device:", DEVICE)


In [ ]:
# Stage 2 - what this notebook tests
#
# NB27 found 25 collapsed evaluations in 350, with 4 of 7 events hitting ALL FIVE
# methods at the identical epoch. Two explanations fit that synchrony:
#
#   H1  BatchNorm running-statistic corruption. Weights stay usable; the stored
#       running mean/var are displaced by an extreme batch, so eval-mode output
#       collapses while batch-statistic output stays healthy.
#   H2  Optimisation-step damage. The weights themselves move to a bad point, so
#       the model is collapsed under BOTH evaluation modes.
#
# Discriminator: at every epoch evaluate the SAME weights twice, once in eval mode
# (running statistics) and once with batch statistics. H1 predicts a large gap.
# H2 predicts both collapse together.
#
# Reproduction targets are the NB27 cells that actually collapsed, plus one clean
# control seed. Same frozen structures, same recipe, same seeds, so the collapse
# epochs should reappear at the same units.

TARGETS = [
    {"architecture": "shallow", "seed": 307, "note": "collapsed at units 4 and 8 for all methods"},
    {"architecture": "deep",    "seed": 401, "note": "collapsed at unit 2 for all methods"},
    {"architecture": "shallow", "seed": 101, "note": "control, no collapse in NB27"},
]
METHODS = ["fisher", "saber_v2"]     # two methods suffice to confirm synchrony
COLLAPSE_B2A = 0.10                  # same threshold used to count NB27 events
print(json.dumps({"targets": TARGETS, "methods": METHODS,
                  "collapse_threshold_benign_to_attack": COLLAPSE_B2A}, indent=2))


In [ ]:
# Stage 3 - frozen configuration, data, teachers
SABER_CFG = yaml.safe_load(open(REPO / "config/saber.yaml"))
E_MAX = {"shallow": 8, "deep": 6}
MIN_W = {"shallow": int(SABER_CFG["groups"]["minimum_remaining_per_layer"]), "deep": 8}
SUBSET_FRACTION = 0.10

TRAIN_LOADER, VAL_LOADER, _TEST_UNUSED, SHALLOW_TEACHER, CLASS_NAMES = load_bridge()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES)
robust_graph = pd.read_csv(R / "14_risk_graph/asvg_edges_robust.csv")
N_CLASSES = len(CLASS_NAMES)


class DeepCNN1D(nn.Module):
    def __init__(self, n_classes=34):
        super().__init__()
        def blk(i, o):
            return [nn.Conv1d(i, o, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(o)]
        self.conv = nn.Sequential(*blk(1, 64), *blk(64, 128), nn.MaxPool1d(2),
                                  *blk(128, 128), *blk(128, 256))
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(256, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


ARCHS = sorted({t["architecture"] for t in TARGETS})
TEACHERS = {}
if "shallow" in ARCHS:
    TEACHERS["shallow"] = SHALLOW_TEACHER.to(DEVICE).eval()
if "deep" in ARCHS:
    _dt = DeepCNN1D(N_CLASSES)
    _dt.load_state_dict(torch.load(REPO / "models/ciciot2023/deepcnn1d_g5_seed0.pt",
                                   map_location="cpu", weights_only=False)["state_dict"])
    TEACHERS["deep"] = _dt.to(DEVICE).eval()

Xv, Yv = VAL_LOADER.dataset.tensors
VAL_Y_ALL = Yv.numpy()
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000]
                       for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
# The per-class construction leaves _idx grouped by class. Eval mode is unaffected, but the
# batch-statistic probe normalises within each chunk, so class-homogeneous chunks would make
# the two modes incomparable. Shuffle once with a fixed seed.
_idx = np.random.default_rng(12345).permutation(_idx)
EX_X = Xv[_idx].to(DEVICE)
EX_Y = VAL_Y_ALL[_idx]
assert len(EX_X) == len(EX_Y)
EXAMPLE_INPUT = Xv[:8].float().to(DEVICE)
_benign = [i for i, c in enumerate(CLASS_NAMES) if "benign" in str(c).lower()]
assert len(_benign) == 1, f"could not identify a unique benign class in {CLASS_NAMES}"
BENIGN_IDX = int(_benign[0])
print("teachers:", list(TEACHERS), "| evaluation rows:", len(_idx),
      "| benign class index:", BENIGN_IDX, f"({CLASS_NAMES[BENIGN_IDX]})")


In [ ]:
# Stage 4 - helpers: dual-mode evaluation and BatchNorm probes
def logits_eval_mode(model):
    """Standard evaluation: BatchNorm uses stored running statistics."""
    model.eval()
    with torch.no_grad():
        return torch.cat([model(EX_X[i:i + 8192]).cpu()
                          for i in range(0, len(EX_X), 8192)]).numpy()


def logits_batch_stats(model):
    """Same weights, but BatchNorm normalises with each batch's own statistics.

    Running statistics are saved and restored, so this probe cannot alter the model.
    """
    saved = {n: (m.running_mean.clone(), m.running_var.clone(), m.momentum,
                 m.num_batches_tracked.clone())
             for n, m in model.named_modules()
             if isinstance(m, nn.BatchNorm1d) and m.running_mean is not None}
    model.train()                                   # batch statistics path
    for _, m in model.named_modules():
        if isinstance(m, nn.BatchNorm1d):
            m.momentum = 0.0                        # freeze running-stat updates
    with torch.no_grad():
        out = torch.cat([model(EX_X[i:i + 8192]).cpu()
                         for i in range(0, len(EX_X), 8192)]).numpy()
    for n, m in model.named_modules():
        if n in saved:
            rm, rv, mom, nbt = saved[n]
            m.running_mean.copy_(rm); m.running_var.copy_(rv)
            m.momentum = mom; m.num_batches_tracked.copy_(nbt)
    model.eval()
    return out


def audit(logits, architecture):
    a = full_model_audit(logits, EX_Y, taxonomy, DEFAULT_COST_PROFILES)
    aw, _ = action_weighted_boundary_inversion_rate(
        T_LOGITS[architecture], logits, EX_Y, robust_graph)
    a["awbir"] = float(aw)
    return a


def bn_state(model):
    """Summary of BatchNorm running statistics: displacement shows up here."""
    means, varis = [], []
    for _, m in model.named_modules():
        if isinstance(m, nn.BatchNorm1d):
            means.append(m.running_mean.detach().cpu().numpy())
            varis.append(m.running_var.detach().cpu().numpy())
    if not means:
        return {}
    mean = np.concatenate(means); var = np.concatenate(varis)
    return {"bn_mean_absmax": float(np.abs(mean).max()),
            "bn_mean_l2": float(np.linalg.norm(mean)),
            "bn_var_max": float(var.max()),
            "bn_var_min": float(var.min()),
            "bn_var_l2": float(np.linalg.norm(var))}


T_LOGITS = {a: logits_eval_mode(m) for a, m in TEACHERS.items()}

# Runtime proof that the batch-statistic probe cannot alter a model: eval-mode output
# must be bit-identical before and after running it.
for _a, _m in TEACHERS.items():
    _buffers_before = {n: (b.running_mean.clone(), b.running_var.clone())
                       for n, b in _m.named_modules() if isinstance(b, nn.BatchNorm1d)}
    _before = logits_eval_mode(_m)
    _probe_out = logits_batch_stats(_m)
    _after = logits_eval_mode(_m)
    # the substantive claim: every BatchNorm buffer is byte-for-byte restored
    for n, b in _m.named_modules():
        if isinstance(b, nn.BatchNorm1d):
            rm, rv = _buffers_before[n]
            assert torch.equal(b.running_mean, rm), f"{_a}/{n}: running_mean not restored"
            assert torch.equal(b.running_var, rv), f"{_a}/{n}: running_var not restored"
    assert np.allclose(_before, _after, atol=1e-5), f"probe altered the {_a} model output"
    assert not _m.training, f"{_a} model left in train mode"
    # the probe must actually do something different, or it cannot discriminate H1 from H2
    assert not np.allclose(_before, _probe_out, atol=1e-6), (
        f"{_a}: batch-statistic probe is indistinguishable from eval mode")
print("probe verified non-destructive and discriminative on:", list(TEACHERS))


def removed_path(architecture, method):
    if architecture == "shallow":
        return R / f"17b_calibrated_checkpoint_freeze/{method}_r40cal_removed_groups.csv"
    return R / f"20b_depth_checkpoint_freeze/{method}_minimal_r40_removed_groups.csv"


def raw_student(architecture, method):
    removed = pd.read_csv(removed_path(architecture, method))
    prune_map = {str(layer): sorted(grp["channel_index"].astype(int).tolist())
                 for layer, grp in removed.groupby("module_path")}
    student, _ = prune_cnn1d_channels(
        TEACHERS[architecture], prune_map, EXAMPLE_INPUT,
        minimum_remaining_per_layer=MIN_W[architecture])
    return student.to(DEVICE)


_train_y = TRAIN_LOADER.dataset.tensors[1].numpy()
_counts = np.bincount(_train_y, minlength=N_CLASSES)
_w = np.zeros_like(_counts, dtype=np.float64)
_w[_counts > 0] = 1.0 / np.sqrt(_counts[_counts > 0])
_w[_counts > 0] /= _w[_counts > 0].mean()
CLASS_W = torch.tensor(_w, dtype=torch.float32, device=DEVICE)
N_TRAIN = len(TRAIN_LOADER.dataset)
print("helpers ready")


In [ ]:
# Stage 5 - reproduce the NB27 cells, evaluating both modes every epoch
EPOCH_CSV = OUT / "dual_mode_epochs.csv"
BATCH_CSV = OUT / "batch_diagnostics.csv"
def _complete(frame):
    """Cells holding every unit 0..E_MAX. Interrupted cells are not complete."""
    if frame.empty:
        return set()
    counts = frame.groupby(["architecture", "method", "seed"])["unit"].nunique()
    return {(a, m, s) for (a, m, s), n in counts.items() if n == E_MAX[a] + 1}


if EPOCH_CSV.exists():
    _e = pd.read_csv(EPOCH_CSV)
    _done = _complete(_e)
    _keys = _e.set_index(["architecture", "method", "seed"]).index
    epoch_rows = _e[[k in _done for k in _keys]].to_dict("records")
else:
    epoch_rows, _done = [], set()

if BATCH_CSV.exists():
    _b = pd.read_csv(BATCH_CSV)
    _bkeys = _b.set_index(["architecture", "method", "seed"]).index
    batch_rows = _b[[k in _done for k in _bkeys]].to_dict("records")
else:
    batch_rows = []
print("complete cells:", len(_done), "| reusable epoch rows:", len(epoch_rows))

for target in TARGETS:
    arch, seed = target["architecture"], target["seed"]
    for method in METHODS:
        if (arch, method, seed) in _done:
            print("skip (done):", arch, method, seed); continue
        torch.manual_seed(seed); np.random.seed(seed)
        student = raw_student(arch, method)

        a_eval = audit(logits_eval_mode(student), arch)
        a_batch = audit(logits_batch_stats(student), arch)
        epoch_rows.append({"architecture": arch, "method": method, "seed": seed, "unit": 0,
                           "eval_family_f1": a_eval["family_macro_f1"],
                           "eval_b2a": a_eval["benign_to_attack_rate"],
                           "eval_awbir": a_eval["awbir"],
                           "batch_family_f1": a_batch["family_macro_f1"],
                           "batch_b2a": a_batch["benign_to_attack_rate"],
                           "batch_awbir": a_batch["awbir"],
                           "max_batch_loss": np.nan, "mean_batch_loss": np.nan,
                           "final_batch_benign_fraction": np.nan, **bn_state(student)})

        gen = torch.Generator().manual_seed(seed)
        subset = torch.randperm(N_TRAIN, generator=gen)[: int(N_TRAIN * SUBSET_FRACTION)]
        loader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(TRAIN_LOADER.dataset, subset.tolist()),
            batch_size=1024, shuffle=True,
            generator=torch.Generator().manual_seed(seed))
        optimiser = torch.optim.Adam(student.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss(weight=CLASS_W)

        for unit in range(1, E_MAX[arch] + 1):
            student.train()
            losses, last_benign_fraction = [], np.nan
            for bi, (xb, yb) in enumerate(loader):
                yb = yb.to(DEVICE)
                optimiser.zero_grad()
                loss = criterion(student(xb.float().to(DEVICE)), yb)
                loss.backward(); optimiser.step()
                losses.append(float(loss.item()))
                last_benign_fraction = float((yb == BENIGN_IDX).float().mean().item())
                batch_rows.append({"architecture": arch, "method": method, "seed": seed,
                                   "unit": unit, "batch_index": bi,
                                   "loss": float(loss.item()),
                                   "benign_fraction": last_benign_fraction})

            a_eval = audit(logits_eval_mode(student), arch)
            a_batch = audit(logits_batch_stats(student), arch)
            epoch_rows.append({"architecture": arch, "method": method, "seed": seed, "unit": unit,
                               "eval_family_f1": a_eval["family_macro_f1"],
                               "eval_b2a": a_eval["benign_to_attack_rate"],
                               "eval_awbir": a_eval["awbir"],
                               "batch_family_f1": a_batch["family_macro_f1"],
                               "batch_b2a": a_batch["benign_to_attack_rate"],
                               "batch_awbir": a_batch["awbir"],
                               "max_batch_loss": float(np.max(losses)),
                               "mean_batch_loss": float(np.mean(losses)),
                               "final_batch_benign_fraction": last_benign_fraction,
                               **bn_state(student)})
            flag = "  <-- COLLAPSE (eval mode)" if a_eval["benign_to_attack_rate"] > COLLAPSE_B2A else ""
            print(f"{arch} {method} s{seed} u{unit}: "
                  f"eval b2a={a_eval['benign_to_attack_rate']:.4f} f1={a_eval['family_macro_f1']:.3f} | "
                  f"batch b2a={a_batch['benign_to_attack_rate']:.4f} f1={a_batch['family_macro_f1']:.3f}{flag}")

        pd.DataFrame(epoch_rows).to_csv(EPOCH_CSV, index=False)
        pd.DataFrame(batch_rows).to_csv(BATCH_CSV, index=False)

epochs = pd.DataFrame(epoch_rows)
batches = pd.DataFrame(batch_rows)
print("epoch rows:", len(epochs), "| batch rows:", len(batches))


In [ ]:
# Stage 6 - verdict: H1 (BatchNorm statistics) vs H2 (weights)
epochs = pd.read_csv(OUT / "dual_mode_epochs.csv")
batches = pd.read_csv(OUT / "batch_diagnostics.csv")

post = epochs[epochs["unit"] > 0].copy()
post["collapsed_eval"] = post["eval_b2a"] > COLLAPSE_B2A
post["collapsed_batch"] = post["batch_b2a"] > COLLAPSE_B2A

events = post[post["collapsed_eval"]]
n_events = int(len(events))
n_batch_also = int(events["collapsed_batch"].sum())
reproduced = sorted({(r.architecture, int(r.seed), int(r.unit)) for r in events.itertuples()})

if n_events == 0:
    verdict = "no_collapse_reproduced"
elif n_batch_also == 0:
    verdict = "H1_batchnorm_running_statistics"
elif n_batch_also == n_events:
    verdict = "H2_weight_damage"
else:
    verdict = "mixed"

clean = post[~post["collapsed_eval"]]

# Unit 0 is the raw pruned model, known to be collapsed in eval mode. It is a free positive
# control: if the batch-statistic probe shows it healthy, displaced running statistics explain
# the raw-pruned collapse too, which is itself a reportable result.
raw = epochs[epochs["unit"] == 0]
raw_control = {
    "eval_b2a_median": float(raw["eval_b2a"].median()) if len(raw) else None,
    "batch_b2a_median": float(raw["batch_b2a"].median()) if len(raw) else None,
    "eval_family_f1_median": float(raw["eval_family_f1"].median()) if len(raw) else None,
    "batch_family_f1_median": float(raw["batch_family_f1"].median()) if len(raw) else None,
} if len(raw) else None
summary = {
    "verdict": verdict,
    "collapse_threshold_benign_to_attack": COLLAPSE_B2A,
    "eval_mode_collapses": n_events,
    "of_which_also_collapsed_under_batch_statistics": n_batch_also,
    "post_recovery_evaluations": int(len(post)),
    "reproduced_collapse_epochs": [{"architecture": a, "seed": s, "unit": u}
                                   for a, s, u in reproduced],
    "raw_pruned_control_unit0": raw_control,
    "median_family_f1_when_eval_collapsed": {
        "eval_mode": float(events["eval_family_f1"].median()) if n_events else None,
        "batch_statistics": float(events["batch_family_f1"].median()) if n_events else None},
    "bn_running_var_max": {
        "collapsed_epochs": (float(events["bn_var_max"].median())
                             if n_events and "bn_var_max" in events else None),
        "healthy_epochs": (float(clean["bn_var_max"].median())
                           if len(clean) and "bn_var_max" in clean else None)},
    "bn_running_mean_absmax": {
        "collapsed_epochs": (float(events["bn_mean_absmax"].median())
                             if n_events and "bn_mean_absmax" in events else None),
        "healthy_epochs": (float(clean["bn_mean_absmax"].median())
                           if len(clean) and "bn_mean_absmax" in clean else None)},
    "max_batch_loss": {
        "collapsed_epochs": float(events["max_batch_loss"].median()) if n_events else None,
        "healthy_epochs": float(clean["max_batch_loss"].median()) if len(clean) else None},
    "interpretation": {
        "H1_batchnorm_running_statistics":
            "weights remain usable; stored running statistics are displaced. Fix: recalibrate "
            "BatchNorm statistics before deployment, or select checkpoints on a safety-gated "
            "per-epoch audit.",
        "H2_weight_damage":
            "the optimisation step itself moved the weights to an unusable point; BatchNorm "
            "recalibration would not help and per-epoch safety-gated checkpointing is required.",
        "mixed": "both mechanisms occur; report per event."},
}
(OUT / "collapse_verdict.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

cols = [c for c in ["architecture", "method", "seed", "unit", "eval_b2a", "batch_b2a",
                    "eval_family_f1", "batch_family_f1", "bn_var_max", "max_batch_loss"]
        if c in events.columns]
if n_events:
    print("\ncollapsed epochs, both evaluation modes:")
    print(events[cols].round(4).to_string(index=False))


In [ ]:
# Stage 7 - figures
epochs = pd.read_csv(OUT / "dual_mode_epochs.csv")
cells_present = epochs[["architecture", "seed"]].drop_duplicates().values.tolist()

fig, axes = plt.subplots(1, len(cells_present), figsize=(4.8 * len(cells_present), 3.4),
                         squeeze=False)
for ax, (arch, seed) in zip(axes[0], cells_present):
    sub = epochs[(epochs["architecture"] == arch) & (epochs["seed"] == seed)]
    for method in sorted(sub["method"].unique()):
        s = sub[sub["method"] == method].sort_values("unit")
        ax.plot(s["unit"], s["eval_b2a"], marker="o", lw=1.3, label=f"{method} eval")
        ax.plot(s["unit"], s["batch_b2a"], marker="s", ls="--", lw=1.1,
                label=f"{method} batch-stats")
    ax.axhline(COLLAPSE_B2A, color="0.4", ls=":", lw=0.9)
    ax.set_yscale("symlog", linthresh=1e-3)
    ax.set_title(f"{arch}, seed {seed}")
    ax.set_xlabel("recovery unit"); ax.set_ylabel("benign to attack rate")
    ax.legend(fontsize=6)
fig.tight_layout(); fig.savefig(OUT / "dual_mode_benign_false_alerts.png", dpi=200); plt.show()

fig, axes = plt.subplots(1, len(cells_present), figsize=(4.8 * len(cells_present), 3.2),
                         squeeze=False)
for ax, (arch, seed) in zip(axes[0], cells_present):
    sub = epochs[(epochs["architecture"] == arch) & (epochs["seed"] == seed)]
    for method in sorted(sub["method"].unique()):
        s = sub[sub["method"] == method].sort_values("unit")
        ax.plot(s["unit"], s["bn_var_max"], marker="o", lw=1.3, label=method)
    ax.set_yscale("log")
    ax.set_title(f"{arch}, seed {seed}: BatchNorm running variance (max)")
    ax.set_xlabel("recovery unit"); ax.set_ylabel("max running var")
    ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "batchnorm_running_variance.png", dpi=200); plt.show()
print("figures written ->", OUT)
